In [39]:
import imaplib
import email
import re 
import pandas as pd 
import io


In [5]:
GMAIL_USER = "******@gmail.com"
GMAIL_APP_PASSWORD = "***********"

mail = imaplib.IMAP4_SSL("imap.gmail.com")
mail.login(GMAIL_USER, GMAIL_APP_PASSWORD)

print("Conection Success")

Conectado correctamente


In [13]:
# I check if the connection was correct with the folders
status, folders = mail.list()

if status == "OK":
    for folder in folders:
        print(folder.decode())
else:
    print("Folders Not Found")

(\HasNoChildren) "/" "Airflow Alerts"
(\HasNoChildren) "/" "INBOX"
(\HasNoChildren) "/" "PRUEBASMAKE"
(\HasChildren \Noselect) "/" "[Gmail]"
(\Drafts \HasNoChildren) "/" "[Gmail]/Borradores"
(\Flagged \HasNoChildren) "/" "[Gmail]/Destacados"
(\HasNoChildren \Sent) "/" "[Gmail]/Enviados"
(\HasNoChildren \Important) "/" "[Gmail]/Importantes"
(\HasNoChildren \Trash) "/" "[Gmail]/Papelera"
(\HasNoChildren \Junk) "/" "[Gmail]/Spam"
(\All \HasNoChildren) "/" "[Gmail]/Todos"


In [47]:
# Select a Folder. In my case is "PRUEBASMAKE"
mail.select("PRUEBASMAKE")

#We can count the emails 

status, messages = mail.search(None, 'UNSEEN') # We filter the emails unreaded 
# UNSEEN --> Unreaded
# SEEN --> Readed
# (FROM "mailremitente@gmail.com") --> To
# '(SUBJECT "Reporte")' --> Subject
# '(SINCE "01-Feb-2026")' --> From a date
# '(BEFORE "15-Feb-2026")' --> To a date
# Filters combinated:  '(UNSEEN SUBJECT "Reporte")'


if status == "OK":
    email_ids = messages[0].split()
    print(f"Se encontraron {len(email_ids)} mails.")
else:
    print("No emails find")

Se encontraron 1 mails.


## Parser y extracción de los cuerpos del mail
##### Vamos a Parsear el asunto y el cuerpo del mail para extraer lo necesario

In [49]:
# I extract the body and put into a df
data = []

for email_id in email_ids:

    status, msg_data = mail.fetch(email_id, "(RFC822)")

    if status != "OK":
        print(f"Error al traer el mail con ID {email_id.decode()}")
        errores += 1
        continue

    raw_email = msg_data[0][1]
    msg = email.message_from_bytes(raw_email)

# Parser Subject
    pattern_subject = r'Fwd:\s(.*?)(?=\s\d{1,2}-\d{1,2}(?:-\d{2,4})?)'
    subject_match = re.search(pattern_subject, msg["subject"])
    subject_clean = subject_match.group(1) if subject_match else msg["subject"]

    remitente = msg["from"]
    date = msg["date"]

    print("Iterando mail:")
    print(subject_clean)
    print(remitente)
    print(date)

# Filter emails witout attachment 
    tiene_adjunto_tabular = False

    if msg.is_multipart():
        for part in msg.walk():
            content_disposition = str(part.get("Content-Disposition"))
            filename = part.get_filename()

            if "attachment" in content_disposition and filename:
                if filename.lower().endswith((".csv", ".xlsx", ".xls")):
                    tiene_adjunto_tabular = True
                    break

    if tiene_adjunto_tabular:
        print("Mail con adjunto CSV/Excel detectado → se omite body")
        continue  

# Extract the body 
    body = None

    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                body = part.get_payload(decode=True).decode(errors="ignore")
                break
    else:
        body = msg.get_payload(decode=True).decode(errors="ignore")

    if not body:
        continue

# Parser the body with the info that I need 
    pattern_body = r'\b(?:[A-Za-z]{2,6}\d+[a-zA-Z]{0,3}|\d{9,})\b'
    matches = re.findall(pattern_body, body)

    print(f"Se encontraron {len(matches)} tracking numbers en este mail")

    for tracking in matches:
        data.append({
            "tracking": tracking,
            "subject": subject_clean,
            "from": remitente,
            "date": date
        })

df = pd.DataFrame(data)

print(df.head())

Iterando mail:
Mail America Load
Josefina Basterra <jbasterra@mailamericas.com>
Thu, 19 Jun 2025 16:08:56 -0300
Se encontraron 118 tracking numbers en este mail
      tracking            subject  \
0  44909642603  Mail America Load   
1  44938123515  Mail America Load   
2  44948583894  Mail America Load   
3  44950155145  Mail America Load   
4  44953508074  Mail America Load   

                                             from  \
0  Josefina Basterra <jbasterra@mailamericas.com>   
1  Josefina Basterra <jbasterra@mailamericas.com>   
2  Josefina Basterra <jbasterra@mailamericas.com>   
3  Josefina Basterra <jbasterra@mailamericas.com>   
4  Josefina Basterra <jbasterra@mailamericas.com>   

                              date  
0  Thu, 19 Jun 2025 16:08:56 -0300  
1  Thu, 19 Jun 2025 16:08:56 -0300  
2  Thu, 19 Jun 2025 16:08:56 -0300  
3  Thu, 19 Jun 2025 16:08:56 -0300  
4  Thu, 19 Jun 2025 16:08:56 -0300  


In [53]:
# Dowload XLSX
df.to_excel("trackings.xlsx", index=False)